# Linear Regression Baseline — Heat Pump Load Forecasting

Simple baseline: aggregate 5-minute telemetry to daily averages per device, train linear regression to predict daily avg x2, then generate monthly predictions for submission.

In [1]:
import pandas as pd
import numpy as np


DATA_PATH = "data/data.csv"
CHUNK_SIZE = 500_000

FEATURE_COLS = [
    "t1", "t2", "t3", "t4", "t5", "t6", "t7",
    "t8", "t9", "t10", "t11", "t12", "t13",
    "x1", "x3", "deviceType",
]
TARGET_COL = "x2"
ALL_COLS = FEATURE_COLS + [TARGET_COL]

print("Imports OK")

Imports OK


## Cell 2 — Chunked load & daily aggregation (train only)

In [2]:
train_sums = []
train_counts = []

for i, chunk in enumerate(pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False)):
    # Filter train only
    chunk = chunk[chunk["period"] == "train"].copy()
    if len(chunk) == 0:
        continue

    # Parse timedate and extract date
    chunk["timedate"] = pd.to_datetime(
        chunk["timedate"].str.replace(" UTC", "", regex=False), utc=True
    )
    chunk["date"] = chunk["timedate"].dt.date

    # Only keep columns we need
    cols_present = [c for c in ALL_COLS if c in chunk.columns]
    grp = chunk.groupby(["deviceId", "date"])[cols_present]

    train_sums.append(grp.sum())
    train_counts.append(grp.count())

    if (i + 1) % 5 == 0:
        print(f"  processed {(i + 1) * CHUNK_SIZE:,} rows...")

print("Combining partial aggregations...")
total_sum = pd.concat(train_sums).groupby(level=["deviceId", "date"]).sum()
total_count = pd.concat(train_counts).groupby(level=["deviceId", "date"]).sum()
daily_train = (total_sum / total_count).reset_index()

del train_sums, train_counts, total_sum, total_count

print(f"Daily train aggregation complete: {daily_train.shape}")

  processed 2,500,000 rows...
  processed 5,000,000 rows...
  processed 7,500,000 rows...
  processed 10,000,000 rows...
  processed 12,500,000 rows...
  processed 15,000,000 rows...
  processed 17,500,000 rows...
  processed 20,000,000 rows...
  processed 22,500,000 rows...
  processed 25,000,000 rows...
  processed 27,500,000 rows...
  processed 30,000,000 rows...
  processed 32,500,000 rows...
  processed 35,000,000 rows...
  processed 37,500,000 rows...
  processed 40,000,000 rows...
  processed 42,500,000 rows...
  processed 45,000,000 rows...
  processed 47,500,000 rows...
  processed 50,000,000 rows...
  processed 52,500,000 rows...
  processed 55,000,000 rows...
  processed 57,500,000 rows...
  processed 60,000,000 rows...
  processed 62,500,000 rows...
Combining partial aggregations...
Daily train aggregation complete: (122310, 19)


In [10]:
daily_train.describe().T

,count,mean,std,min,25%,50%,75%,max
t1,122310.0,2.803913e-01,0.032390,0.000000,0.260833,0.276701,0.297535,1.000000
t2,122310.0,4.533377e-02,0.005849,0.030000,0.040000,0.046021,0.050000,0.160000
t3,122310.0,4.680033e-03,0.062619,0.000000,0.000000,0.000000,0.000000,1.000000
t4,122310.0,3.914938e-01,0.047530,0.000000,0.363507,0.387482,0.418750,1.000000
t5,122310.0,4.354056e-01,0.037649,0.250000,0.415208,0.436944,0.458507,1.000000
t6,122310.0,4.155492e-01,0.032826,0.249340,0.398368,0.418750,0.437150,1.000000
t7,122310.0,2.104008e-01,0.034406,0.110069,0.203229,0.208229,0.213264,0.473333
t8,122310.0,5.142527e-01,0.021561,0.400000,0.499792,0.514896,0.529236,0.580625
t9,122310.0,3.911278e-01,0.068621,0.000000,0.365035,0.388333,0.408630,0.900000
t10,122310.0,2.065640e-01,0.004190,0.180000,0.202361,0.207396,0.210000,0.220833


## Cell 3 — Inspect daily aggregated dataframe

In [3]:
print(f"Shape: {daily_train.shape}")
print(f"\nNull counts:")
print(daily_train.isnull().sum())
print(f"\nUnique (deviceId, date) pairs: {daily_train[['deviceId', 'date']].drop_duplicates().shape[0]}")
daily_train.head()

Shape: (122310, 19)

Null counts:
deviceId      0
date          0
t1            0
t2            0
t3            0
t4            0
t5            0
t6            0
t7            0
t8            0
t9            0
t10           0
t11           0
t12           0
t13           0
x1            0
x3            0
deviceType    0
x2            0
dtype: int64

Unique (deviceId, date) pairs: 122310


,deviceId,date,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,t11,t12,t13,x1,x3,deviceType,x2
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01,0.307118,0.050000,0.0,0.393854,0.442396,0.431250,0.206076,0.507778,0.382951,0.210833,0.209028,0.066771,0.068333,0.0,8.0,19.0,0.102980
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-02,0.307639,0.050000,0.0,0.387778,0.446528,0.434444,0.207917,0.513368,0.386424,0.210000,0.209826,0.067604,0.069653,0.0,8.0,19.0,0.102154
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-03,0.303993,0.050000,0.0,0.389236,0.448715,0.437569,0.206736,0.518056,0.390069,0.210000,0.209931,0.068750,0.070174,0.0,8.0,19.0,0.110116
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-04,0.300389,0.047845,0.0,0.398269,0.457951,0.444912,0.207244,0.520247,0.397350,0.210000,0.209929,0.069682,0.070389,0.0,8.0,19.0,0.123136
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-05,0.300000,0.050000,0.0,0.398229,0.457431,0.444201,0.205833,0.523854,0.396215,0.210000,0.209826,0.069549,0.070139,0.0,8.0,19.0,0.129621


## Cell 4 — Train Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

feature_cols_present = [c for c in FEATURE_COLS if c in daily_train.columns]

In [113]:
feature_cols_present = ['t1',
 't2',
 't3',
 't4',
 't5',
 't6',
 't7',
 't8',
 't9',
 't10',
 't11',
 't12',
 't13',
 'x1',
 'x3',
 'deviceType']

In [97]:
train_clean = daily_train.dropna(subset=feature_cols_present + [TARGET_COL])
X_train = train_clean[feature_cols_present].values
y_train = train_clean[TARGET_COL].values

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
train_mae = mean_absolute_error(y_train, y_pred_train)
print(f"Train MAE (in-sample): {train_mae:.6f}")

print("\nCoefficients:")
for feat, coef in zip(feature_cols_present, model.coef_):
    print(f"  {feat:>12s}: {coef:.6f}")
print(f"  {'intercept':>12s}: {model.intercept_:.6f}")

Train MAE (in-sample): 0.054573

Coefficients:
            t1: -0.150867
            t2: -0.756628
            t3: 0.018848
            t4: 0.476504
            t5: 2.774909
            t6: -2.641620
            t7: 0.269315
            t8: 1.429245
            t9: 0.079012
           t10: -3.573991
           t11: -4.682080
           t12: -1.641452
           t13: 0.861022
            x1: -13.804253
            x3: 0.001452
    deviceType: 0.017310
     intercept: 0.629148


In [114]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Dictionary to store a unique model for each device type
models_per_type = {}
train_info = []

unique_types = daily_train['deviceType'].unique()

for d_type in unique_types:
    # Filter data for this specific device type
    df_type = daily_train[daily_train['deviceType'] == d_type].dropna(subset=feature_cols_present + [TARGET_COL])

    if len(df_type) < 5:  # Minimum samples check
        print(f"Skipping Device Type {d_type}: Not enough data.")
        continue

    X_t = df_type[feature_cols_present].values
    y_t = df_type[TARGET_COL].values

    # Initialize and train standard Linear Regression
    model = LinearRegression()
    model.fit(X_t, y_t)

    # Store the model object
    models_per_type[d_type] = model

    # Calculate in-sample MAE for reporting
    y_p = model.predict(X_t)
    mae = mean_absolute_error(y_t, y_p)
    train_info.append({'Type': d_type, 'MAE': mae, 'Samples': len(df_type)})

# Display summary
print(f"{'Device Type':<15} | {'Train MAE':<12} | {'Sample Count'}")
print("-" * 45)
for info in train_info:
    print(f"{info['Type']:<15} | {info['MAE']:<12.6f} | {info['Samples']}")

Device Type     | Train MAE    | Sample Count
---------------------------------------------
19.0            | 0.090563     | 35154
11.0            | 0.019689     | 74912
7.0             | 0.008838     | 12244


In [69]:
train_clean["deviceType"].value_counts()

deviceType
11.0    74912
19.0    35154
7.0     12244
Name: count, dtype: int64

In [108]:
import optuna
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score

lasso_models = {}
lasso_best_alphas = {}

for d_type in daily_train['deviceType'].unique():
    # Filter data for this device
    df_type = daily_train[daily_train['deviceType'] == d_type].dropna(subset=feature_cols_present + [TARGET_COL])
    if len(df_type) < 20: continue # Skip if data is too sparse for CV

    X_t, y_t = df_type[feature_cols_present].values, df_type[TARGET_COL].values

    def objective(trial):
        # Search alpha on a log scale
        alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)
        # We increase max_iter to ensure Lasso converges with small alphas
        model = Lasso(alpha=alpha, max_iter=5000, random_state=42)
        model.fit(X_t, y_t)
        # scoring is 'neg_mean_absolute_error' because we want to minimize MAE
        preds = model.predict(X_t)
        return mean_absolute_error(y_t, preds)

    print(f"Optimizing Lasso for Device: {d_type}")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=300, show_progress_bar=False)

    # Train the final Lasso model for this device using the best alpha
    best_alpha = study.best_params['alpha']
    final_model = Lasso(alpha=best_alpha, max_iter=5000).fit(X_t, y_t)

    lasso_models[d_type] = final_model
    lasso_best_alphas[d_type] = best_alpha

print("\nBest Lasso Alphas found:", lasso_best_alphas)

[I 2026-03-14 19:58:45,249] A new study created in memory with name: no-name-3b35e0da-bd1d-418d-a6d2-4225d1346a82
[I 2026-03-14 19:58:45,252] Trial 0 finished with value: 0.1680971070540161 and parameters: {'alpha': 1.6422656647711475}. Best is trial 0 with value: 0.1680971070540161.
[I 2026-03-14 19:58:45,256] Trial 1 finished with value: 0.1680971070540161 and parameters: {'alpha': 0.12894104473129148}. Best is trial 0 with value: 0.1680971070540161.
[I 2026-03-14 19:58:45,263] Trial 2 finished with value: 0.09195853631476762 and parameters: {'alpha': 2.6466461212041496e-05}. Best is trial 2 with value: 0.09195853631476762.
[I 2026-03-14 19:58:45,266] Trial 3 finished with value: 0.1680971070540161 and parameters: {'alpha': 1.405881015935189}. Best is trial 2 with value: 0.09195853631476762.
[I 2026-03-14 19:58:45,273] Trial 4 finished with value: 0.09108769015369172 and parameters: {'alpha': 1.1000152951517555e-05}. Best is trial 4 with value: 0.09108769015369172.
[I 2026-03-14 19:5

Optimizing Lasso for Device: 19.0


[I 2026-03-14 19:58:45,454] Trial 38 finished with value: 0.0915671363115249 and parameters: {'alpha': 2.0007561841507614e-05}. Best is trial 22 with value: 0.09105686026669048.
[I 2026-03-14 19:58:45,463] Trial 39 finished with value: 0.09613728666800274 and parameters: {'alpha': 8.139740328343758e-05}. Best is trial 22 with value: 0.09105686026669048.
[I 2026-03-14 19:58:45,466] Trial 40 finished with value: 0.11415349309980595 and parameters: {'alpha': 0.0015084872788427232}. Best is trial 22 with value: 0.09105686026669048.
[I 2026-03-14 19:58:45,473] Trial 41 finished with value: 0.09107736797861218 and parameters: {'alpha': 1.0790074134472351e-05}. Best is trial 22 with value: 0.09105686026669048.
[I 2026-03-14 19:58:45,480] Trial 42 finished with value: 0.09151295074784785 and parameters: {'alpha': 1.9059793655518816e-05}. Best is trial 22 with value: 0.09105686026669048.
[I 2026-03-14 19:58:45,486] Trial 43 finished with value: 0.09114780835619851 and parameters: {'alpha': 1.22

Optimizing Lasso for Device: 11.0


[I 2026-03-14 19:58:47,286] Trial 37 finished with value: 0.02040979424810576 and parameters: {'alpha': 3.491843780485929e-05}. Best is trial 31 with value: 0.019822453551504044.
[I 2026-03-14 19:58:47,290] Trial 38 finished with value: 0.05831282177726149 and parameters: {'alpha': 0.2859567503399078}. Best is trial 31 with value: 0.019822453551504044.
[I 2026-03-14 19:58:47,295] Trial 39 finished with value: 0.02050910434820649 and parameters: {'alpha': 0.00010762923332191633}. Best is trial 31 with value: 0.019822453551504044.
[I 2026-03-14 19:58:47,299] Trial 40 finished with value: 0.020437100854737714 and parameters: {'alpha': 2.5071033789422114e-05}. Best is trial 31 with value: 0.019822453551504044.
[I 2026-03-14 19:58:47,309] Trial 41 finished with value: 0.019872557363246404 and parameters: {'alpha': 1.1342462571882354e-05}. Best is trial 31 with value: 0.019822453551504044.
[I 2026-03-14 19:58:47,318] Trial 42 finished with value: 0.01982676777935515 and parameters: {'alpha':

Optimizing Lasso for Device: 7.0


[I 2026-03-14 19:58:49,510] Trial 108 finished with value: 0.00918423746960147 and parameters: {'alpha': 1.3231912121615289e-05}. Best is trial 32 with value: 0.0091676302021273.
[I 2026-03-14 19:58:49,512] Trial 109 finished with value: 0.009221664925612565 and parameters: {'alpha': 1.7822245458903613e-05}. Best is trial 32 with value: 0.0091676302021273.
[I 2026-03-14 19:58:49,514] Trial 110 finished with value: 0.009329692133679866 and parameters: {'alpha': 2.5985478728749516e-05}. Best is trial 32 with value: 0.0091676302021273.
[I 2026-03-14 19:58:49,516] Trial 111 finished with value: 0.009167650743836646 and parameters: {'alpha': 1.0021436350225627e-05}. Best is trial 32 with value: 0.0091676302021273.
[I 2026-03-14 19:58:49,518] Trial 112 finished with value: 0.009194290723671271 and parameters: {'alpha': 1.4669324452388389e-05}. Best is trial 32 with value: 0.0091676302021273.
[I 2026-03-14 19:58:49,521] Trial 113 finished with value: 0.009222146717083702 and parameters: {'alp


Best Lasso Alphas found: {np.float64(19.0): 1.0001370633335225e-05, np.float64(11.0): 1.0000293643965711e-05, np.float64(7.0): 1.0004767844784342e-05}


In [109]:
from sklearn.linear_model import Ridge

ridge_models = {}
ridge_best_alphas = {}

for d_type in daily_train['deviceType'].unique():
    df_type = daily_train[daily_train['deviceType'] == d_type].dropna(subset=feature_cols_present + [TARGET_COL])
    if len(df_type) < 20: continue

    X_t, y_t = df_type[feature_cols_present].values, df_type[TARGET_COL].values

    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-3, 100.0, log=True)
        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_t, y_t)
        preds = model.predict(X_t)
        return mean_absolute_error(y_t, preds)

    print(f"Optimizing Ridge for Device: {d_type}")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=300, show_progress_bar=False)

    best_alpha = study.best_params['alpha']
    ridge_models[d_type] = Ridge(alpha=best_alpha).fit(X_t, y_t)
    ridge_best_alphas[d_type] = best_alpha

print("\nBest Ridge Alphas found:", ridge_best_alphas)

[I 2026-03-14 19:59:01,679] A new study created in memory with name: no-name-d9cb1818-832d-43bc-bb68-abef88038941
[I 2026-03-14 19:59:01,684] Trial 0 finished with value: 0.09095064107475483 and parameters: {'alpha': 0.04753733452001211}. Best is trial 0 with value: 0.09095064107475483.
[I 2026-03-14 19:59:01,687] Trial 1 finished with value: 0.09078349576770292 and parameters: {'alpha': 0.027898279451983587}. Best is trial 1 with value: 0.09078349576770292.
[I 2026-03-14 19:59:01,690] Trial 2 finished with value: 0.09072676054056673 and parameters: {'alpha': 0.02147177015648667}. Best is trial 2 with value: 0.09072676054056673.
[I 2026-03-14 19:59:01,692] Trial 3 finished with value: 0.09952302671150454 and parameters: {'alpha': 3.1591397076051004}. Best is trial 2 with value: 0.09072676054056673.
[I 2026-03-14 19:59:01,695] Trial 4 finished with value: 0.10054204287683427 and parameters: {'alpha': 4.444336467606881}. Best is trial 2 with value: 0.09072676054056673.
[I 2026-03-14 19:5

Optimizing Ridge for Device: 19.0


[I 2026-03-14 19:59:01,880] Trial 71 finished with value: 0.09056882440507696 and parameters: {'alpha': 0.0010264127587983295}. Best is trial 62 with value: 0.09056875011568877.
[I 2026-03-14 19:59:01,883] Trial 72 finished with value: 0.09057131156929826 and parameters: {'alpha': 0.0014810716812187926}. Best is trial 62 with value: 0.09056875011568877.
[I 2026-03-14 19:59:01,887] Trial 73 finished with value: 0.09056891387630864 and parameters: {'alpha': 0.0010432309716895}. Best is trial 62 with value: 0.09056875011568877.
[I 2026-03-14 19:59:01,890] Trial 74 finished with value: 0.09057579102556475 and parameters: {'alpha': 0.0022479167185133035}. Best is trial 62 with value: 0.09056875011568877.
[I 2026-03-14 19:59:01,893] Trial 75 finished with value: 0.09056875225143907 and parameters: {'alpha': 0.0010128487256609363}. Best is trial 62 with value: 0.09056875011568877.
[I 2026-03-14 19:59:01,896] Trial 76 finished with value: 0.09059526330857548 and parameters: {'alpha': 0.0052268

Optimizing Ridge for Device: 11.0


[I 2026-03-14 19:59:02,711] Trial 56 finished with value: 0.01967040785533188 and parameters: {'alpha': 0.015802447144469108}. Best is trial 48 with value: 0.01964888016768004.
[I 2026-03-14 19:59:02,714] Trial 57 finished with value: 0.01968008027677113 and parameters: {'alpha': 0.4016571699085211}. Best is trial 48 with value: 0.01964888016768004.
[I 2026-03-14 19:59:02,718] Trial 58 finished with value: 0.019654233807712838 and parameters: {'alpha': 0.1942236991170673}. Best is trial 48 with value: 0.01964888016768004.
[I 2026-03-14 19:59:02,721] Trial 59 finished with value: 0.019648843959446015 and parameters: {'alpha': 0.10969377019334087}. Best is trial 59 with value: 0.019648843959446015.
[I 2026-03-14 19:59:02,725] Trial 60 finished with value: 0.019657542772400973 and parameters: {'alpha': 0.038689702175812114}. Best is trial 59 with value: 0.019648843959446015.
[I 2026-03-14 19:59:02,729] Trial 61 finished with value: 0.019648847785364784 and parameters: {'alpha': 0.10815409

Optimizing Ridge for Device: 7.0


[I 2026-03-14 19:59:03,928] Trial 141 finished with value: 0.00878698395176095 and parameters: {'alpha': 0.011632904105297043}. Best is trial 89 with value: 0.008786905404588303.
[I 2026-03-14 19:59:03,929] Trial 142 finished with value: 0.008786903751214836 and parameters: {'alpha': 0.012621599763420126}. Best is trial 142 with value: 0.008786903751214836.
[I 2026-03-14 19:59:03,931] Trial 143 finished with value: 0.00878864645147272 and parameters: {'alpha': 0.008834118551016977}. Best is trial 142 with value: 0.008786903751214836.
[I 2026-03-14 19:59:03,933] Trial 144 finished with value: 0.008787526981224073 and parameters: {'alpha': 0.014648693443229128}. Best is trial 142 with value: 0.008786903751214836.
[I 2026-03-14 19:59:03,934] Trial 145 finished with value: 0.008786945115645812 and parameters: {'alpha': 0.012937347962402384}. Best is trial 142 with value: 0.008786903751214836.
[I 2026-03-14 19:59:03,936] Trial 146 finished with value: 0.008797417660320436 and parameters: {'


Best Ridge Alphas found: {np.float64(19.0): 0.001000029674436957, np.float64(11.0): 0.11121856186497943, np.float64(7.0): 0.0123871357973555}


In [115]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

train_comparison = []

# Get the list of device types that have both models
device_types = set(lasso_models.keys()).intersection(set(ridge_models.keys()))

for d_type in device_types:
    # Filter train data for this device type
    train_subset = daily_train[daily_train['deviceType'] == d_type].dropna(subset=feature_cols_present + [TARGET_COL])

    if len(train_subset) == 0:
        continue

    X_t = train_subset[feature_cols_present].values
    y_t = train_subset[TARGET_COL].values

    # Lasso In-Sample Metrics
    l_pred = lasso_models[d_type].predict(X_t)
    l_mae = mean_absolute_error(y_t, l_pred)
    l_mse = mean_squared_error(y_t, l_pred)

    # Ridge In-Sample Metrics
    r_pred = ridge_models[d_type].predict(X_t)
    r_mae = mean_absolute_error(y_t, r_pred)
    r_mse = mean_squared_error(y_t, r_pred)

    # Track who wins on training data
    best_mae = "Lasso" if l_mae < r_mae else "Ridge"

    train_comparison.append({
        'DeviceType': d_type,
        'Samples': len(train_subset),
        'Lasso_MAE': l_mae,
        'Ridge_MAE': r_mae,
        'Lasso_MSE': l_mse,
        'Ridge_MSE': r_mse,
        'Best_Train_Model': best_mae
    })

# Create DataFrame
train_comp_df = pd.DataFrame(train_comparison)

# Style to highlight the lower error in each row
def highlight_min(s):
    is_min = s == s.min()
    return ['background-color: #e3f2fd' if v else '' for v in is_min]

print("In-Sample Training Comparison (Lasso vs Ridge) per Device Type:")
display(train_comp_df.style.apply(highlight_min, subset=['Lasso_MAE', 'Ridge_MAE'], axis=1)
                    .format(precision=6))

# Tally the results
print(f"\nTraining Set Winners (MAE):")
print(train_comp_df['Best_Train_Model'].value_counts())

In-Sample Training Comparison (Lasso vs Ridge) per Device Type:


,DeviceType,Samples,Lasso_MAE,Ridge_MAE,Lasso_MSE,Ridge_MSE,Best_Train_Model
0,19.000000,35154,0.091039,0.090569,0.011979,0.011900,Ridge
1,11.000000,74912,0.019815,0.019649,0.000775,0.000756,Ridge
2,7.000000,12244,0.009168,0.008787,0.000196,0.000183,Ridge



Training Set Winners (MAE):
Best_Train_Model
Ridge    3
Name: count, dtype: int64


In [110]:
comparison_rows = []
for d_type in lasso_models.keys():
    lasso_coefs = lasso_models[d_type].coef_
    # Count how many features Lasso kept (non-zero)
    active_features = sum(lasso_coefs != 0)

    comparison_rows.append({
        'DeviceType': d_type,
        'Lasso_Alpha': lasso_best_alphas[d_type],
        'Ridge_Alpha': ridge_best_alphas[d_type],
        'Features_Kept_By_Lasso': active_features
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,DeviceType,Lasso_Alpha,Ridge_Alpha,Features_Kept_By_Lasso
0,19.0,0.00001,0.001000,10
1,11.0,0.00001,0.111219,10
2,7.0,0.00001,0.012387,6


In [90]:
deletion_report = []

for d_type, model in lasso_models.items():
    # Identify which features have a 0 coefficient
    # We use a small tolerance (1e-10) just in case of tiny numerical noise
    zero_features = [
        feat for feat, coef in zip(feature_cols_present, model.coef_)
        if abs(coef) < 1e-10
    ]

    # Identify which features are actually being used
    kept_features = [
        feat for feat, coef in zip(feature_cols_present, model.coef_)
        if abs(coef) >= 1e-10
    ]

    deletion_report.append({
        'DeviceType': d_type,
        'Alpha': lasso_best_alphas[d_type],
        'Total_Features': len(feature_cols_present),
        'Kept_Count': len(kept_features),
        'Deleted_Count': len(zero_features),
        'Features_To_Delete': ", ".join(zero_features) if zero_features else "None"
    })

# Convert to DataFrame for a clean view
deletion_df = pd.DataFrame(deletion_report)
display(deletion_df)

,DeviceType,Alpha,Total_Features,Kept_Count,Deleted_Count,Features_To_Delete
0,19.0,0.00001,16,10,6,"t3, t9, t12, t13, x1, deviceType"
1,11.0,0.00001,16,10,6,"t2, t11, t12, t13, x1, deviceType"
2,7.0,0.00001,16,6,10,"t2, t3, t5, t7, t10, t11, t12, t13, x1, device..."


## Cell 5 — Chunked load & daily aggregation (valid + test)

In [5]:
pred_sums = []
pred_counts = []

for i, chunk in enumerate(pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False)):
    # Filter valid + test only
    chunk = chunk[chunk["period"].isin(["valid", "test"])].copy()
    if len(chunk) == 0:
        continue

    chunk["timedate"] = pd.to_datetime(
        chunk["timedate"].str.replace(" UTC", "", regex=False), utc=True
    )
    chunk["date"] = chunk["timedate"].dt.date

    cols_present = [c for c in feature_cols_present if c in chunk.columns]
    grp = chunk.groupby(["deviceId", "date"])[cols_present]

    pred_sums.append(grp.sum())
    pred_counts.append(grp.count())

    if (i + 1) % 5 == 0:
        print(f"  processed {(i + 1) * CHUNK_SIZE:,} rows...")

print("Combining partial aggregations...")
total_sum = pd.concat(pred_sums).groupby(level=["deviceId", "date"]).sum()
total_count = pd.concat(pred_counts).groupby(level=["deviceId", "date"]).sum()
daily_pred_df = (total_sum / total_count).reset_index()

del pred_sums, pred_counts, total_sum, total_count

print(f"Daily valid+test aggregation complete: {daily_pred_df.shape}")

  processed 2,500,000 rows...
  processed 5,000,000 rows...
  processed 7,500,000 rows...
  processed 10,000,000 rows...
  processed 12,500,000 rows...
  processed 15,000,000 rows...
  processed 17,500,000 rows...
  processed 20,000,000 rows...
  processed 22,500,000 rows...
  processed 25,000,000 rows...
  processed 27,500,000 rows...
  processed 30,000,000 rows...
  processed 32,500,000 rows...
  processed 35,000,000 rows...
  processed 37,500,000 rows...
  processed 40,000,000 rows...
  processed 42,500,000 rows...
  processed 45,000,000 rows...
  processed 47,500,000 rows...
  processed 50,000,000 rows...
  processed 52,500,000 rows...
  processed 55,000,000 rows...
  processed 57,500,000 rows...
  processed 60,000,000 rows...
  processed 62,500,000 rows...
Combining partial aggregations...
Daily valid+test aggregation complete: (105529, 18)


## Cell 6 — Predict daily x2, aggregate to monthly

In [38]:
# pred_clean = daily_pred_df.dropna(subset=feature_cols_present).copy()
# X_pred = pred_clean[feature_cols_present].values
# pred_clean["daily_x2_pred"] = model.predict(X_pred)
#
# # Extract year/month from date
# pred_clean["date"] = pd.to_datetime(pred_clean["date"])
# pred_clean["year"] = pred_clean["date"].dt.year
# pred_clean["month"] = pred_clean["date"].dt.month
#
# # Aggregate daily predictions to monthly mean
# monthly_preds = (
#     pred_clean
#     .groupby(["deviceId", "year", "month"])["daily_x2_pred"]
#     .mean()
#     .reset_index()
#     .rename(columns={"daily_x2_pred": "prediction"})
# )
#
# print(f"Monthly predictions shape: {monthly_preds.shape}")
# print(f"Year/month range:")
# print(monthly_preds[["year", "month"]].drop_duplicates().sort_values(["year", "month"]).to_string(index=False))
# monthly_preds.head()

Monthly predictions shape: (3559, 4)
Year/month range:
 year  month
 2025      5
 2025      6
 2025      7
 2025      8
 2025      9
 2025     10


,deviceId,year,month,prediction
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,5,0.225626
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,6,0.164666
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,7,0.165904
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,8,0.163678
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,9,0.199846


# normal regression

In [116]:
# 1. Prepare the dataframe
pred_clean = daily_pred_df.dropna(subset=feature_cols_present).copy()
pred_clean["daily_x2_pred"] = 0.0  # Initialize prediction column

# 2. Apply the correct model for each device type
# We use the 'models_per_type' dictionary created in the previous step

#                   CHANGE MODEL DICT HERE
for d_type, model in ridge_models.items():
    # Find rows matching this device type
    mask = pred_clean["deviceType"] == d_type

    if mask.any():
        X_pred = pred_clean.loc[mask, feature_cols_present].values
        pred_clean.loc[mask, "daily_x2_pred"] = model.predict(X_pred)

# 3. Extract year/month from date
pred_clean["date"] = pd.to_datetime(pred_clean["date"])
pred_clean["year"] = pred_clean["date"].dt.year
pred_clean["month"] = pred_clean["date"].dt.month

# 4. Aggregate daily predictions to monthly mean (as required by the challenge)
monthly_preds = (
    pred_clean
    .groupby(["deviceId", "year", "month"])["daily_x2_pred"]
    .mean()
    .reset_index()
    .rename(columns={"daily_x2_pred": "prediction"})
)

# Summary output
print(f"Monthly predictions shape: {monthly_preds.shape}")
print(f"Year/month range:")
print(monthly_preds[["year", "month"]].drop_duplicates().sort_values(["year", "month"]).to_string(index=False))
monthly_preds.head()

Monthly predictions shape: (3559, 4)
Year/month range:
 year  month
 2025      5
 2025      6
 2025      7
 2025      8
 2025      9
 2025     10


,deviceId,year,month,prediction
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,5,0.109795
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,6,0.033200
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,7,0.021546
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,8,0.020250
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,9,0.087769


## Cell 7 — Generate submission CSV

In [117]:
# Filter to May-Oct 2025 only
submission = monthly_preds[
    (monthly_preds["year"] == 2025) & (monthly_preds["month"].between(5, 10))
].copy()

submission = submission[["deviceId", "year", "month", "prediction"]]

print(f"Submission shape: {submission.shape}")
print(f"Expected: 600 devices × 6 months = 3600 rows")
print(f"Unique devices: {submission['deviceId'].nunique()}")
print(f"Months covered: {sorted(submission['month'].unique())}")
print(f"\nSample:")
print(submission.head(12).to_string(index=False))

OUT_PATH = "data/different_device_ridge_optuna.csv"
submission.to_csv(OUT_PATH, index=False)
print(f"\nSaved → {OUT_PATH}")

Submission shape: (3559, 4)
Expected: 600 devices × 6 months = 3600 rows
Unique devices: 600
Months covered: [np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]

Sample:
                                                        deviceId  year  month  prediction
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      5    0.109795
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      6    0.033200
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      7    0.021546
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      8    0.020250
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      9    0.087769
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025     10    0.190097
005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30fe5a99d7a76b53f9fc9  2025      5    0.049194
005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30fe5a99d7a76b53f9fc9  2025      6    0